In [1]:
import csv
import importlib
import json
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    # from Labs.EnvWrapper import EnvWrapper
    from Labs.EnvCPTWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.FocusWorkers import BaseWorker, EnhWorker, FocusWorker
from RL.DqnHeuWorker import DqnHeuWorker
from RL.A2CWorker import A2CWorker
from RL.A2CHeuWorker import A2CHeuWorker

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders

importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey

cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"
cfg.state_dim = cfg.cache_size * 10 + 2
cfg.action_dim = cfg.cache_size * 5 + 1

cfg.state_dim_base_focus = cfg.cache_size * 10 + 4
cfg.state_dim_enh_focus = cfg.cache_size * 10 + 4
cfg.action_dim_base_focus = cfg.cache_size * 5 + 1

debugger = debugger.debug

In [3]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)

    def build_observation(self, video, tile = None) -> np.ndarray:

        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)

        for idx, (v, t) in enumerate(cache):
            if v == -1:
                continue

            if t == -1:
                x_s[idx] = self.scaled_freq(self.features.video_freq_short.get(v, 0), self.features.video_hist_short.maxlen)
                x_l[idx] = self.scaled_freq(self.features.video_freq_long.get(v, 0), self.features.video_hist_long.maxlen)
            else:
                x_s[idx] = self.scaled_freq(self.features.tile_freq_short.get((v, t), 0), self.features.tile_hist_short.maxlen)
                x_l[idx] = self.scaled_freq(self.features.tile_freq_long.get((v, t), 0), self.features.tile_hist_long.maxlen)

        if tile is None:
            y_s = np.array(
                [self.scaled_freq(self.features.video_freq_short.get(video, 0), self.features.video_hist_short.maxlen)], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.scaled_freq(self.features.video_freq_long.get(video, 0), self.features.video_hist_long.maxlen)], 
                dtype=np.float32
            )
        else:
            y_s = np.array(
                [self.scaled_freq(self.features.tile_freq_short.get((video, tile), 0), self.features.tile_hist_short.maxlen)], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.scaled_freq(self.features.tile_freq_long.get((video, tile), 0), self.features.tile_hist_long.maxlen)], 
                dtype=np.float32
            )

        step_one_hot = int(tile is None)

        # print(f"Observation for video {video}, tile {tile}:")
        # print(f"  Cache state (short-term freq): {x_s}")
        # print(f"  Cache state (long-term freq): {x_l}")
        # print(f"  Request state (short-term freq): {y_s}")
        # print(f"  Request state (long-term freq): {y_l}")
        # print(f"  Step one-hot: {step_one_hot}")

        return np.concatenate(
            [x_s, x_l, y_s, y_l], axis=0
        )

    def scaled_freq(self, count, hist_maxlen, alpha=10.0):
        if hist_maxlen <= 0:
            return 0.0
        return np.log1p(alpha * count) / np.log1p(alpha * hist_maxlen)

    def reset(self):
        
        obs, info = self.env.reset()
        self.features.reset_history()

        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [4]:
def _append_csv_row(csv_path: str, fieldnames: list[str], row: dict) -> None:
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def save_episode_metrics(
    metrics_dir: str,
    ep: int,
    total_reward: float,
    cache_hits: int,
    cache_misses: int,
    agent,
):
    csv_path = os.path.join(metrics_dir, 'episode_metrics.csv')
    fieldnames = [
        'episode',
        'total_reward',
        'cache_hits',
        'cache_misses',
        'hit_rate',
        'epsilon',
        'lr',
    ]

    row = {
        'episode': ep,
        'total_reward': round(float(total_reward), 2),
        'cache_hits': cache_hits,
        'cache_misses': cache_misses,
        'hit_rate': float(cache_hits) / float(cache_hits + cache_misses + 1e-9),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
    }
    _append_csv_row(csv_path, fieldnames, row)

def save_step_metrics(
    metrics_dir: str,
    episode: int,
    episode_step: int,
    global_step: int,
    reward: float,
    agent,
    train_metrics: dict | None,
):
    csv_path = os.path.join(metrics_dir, 'step_metrics.csv')
    fieldnames = [
        'episode',
        'episode_step',
        'global_step',
        'reward',
        'epsilon',
        'lr',
        'train_loss',
        'actor_loss',
        'critic_loss'
    ]

    row = {
        'episode': episode,
        'episode_step': episode_step,
        'global_step': global_step,
        'reward': float(reward),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
        'train_loss': None,
        'actor_loss': None,
        'critic_loss': None
    }

    if train_metrics is not None:
        row.update({
            'train_loss': train_metrics.get('train_loss'),
            'actor_loss': train_metrics.get('actor_loss'),
            'critic_loss': train_metrics.get('critic_loss'),
    })

    _append_csv_row(csv_path, fieldnames, row)

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

In [5]:
def process_missing_layer(
    agent,
    req_state,
    env,
    net_adapter,
    missing,
    transition,
    backhaul_usage: int,
    layer_idx: int,
    tile_idx: int | None,
    is_base: bool,
    bytes_cost: int,
    log_key: str,
) -> int:
    if missing[layer_idx] != 1:
        return backhaul_usage

    if tile_idx is None:
        state = net_adapter.build_observation(req_state["video"])
        tiles = []
    else:
        tile = req_state["viewport"][tile_idx]
        state = net_adapter.build_observation(req_state["video"], tile)
        tiles = [tile]

    selection = agent.select_action(state)
    if isinstance(selection, tuple):
        action = selection[0]
        value = selection[1] if len(selection) > 1 else None
        prob = selection[2] if len(selection) > 2 else None
        entropy = selection[3] if len(selection) > 3 else None
    else:
        action = int(selection)
        value = prob = entropy = None

    transition[layer_idx] = {
        'state': state,
        'action': action,
        'value': value,
        'prob': prob,
        'entropy': entropy
    }

    message = {
        "video": req_state["video"],
        "tiles": tiles,
        "base_req_init": is_base,
        "action_idx": action,
    }
    env.prefetch_fn(env.mec_cache, message)

    if action != 0:
        backhaul_usage += bytes_cost

    debugger.log(f"{log_key}_action", action)
    debugger.log(f"{log_key}_value", value)
    debugger.log(f"{log_key}_prob", prob)
    debugger.log(f"{log_key}_entropy", entropy)

    return backhaul_usage

def select_action(agent, req_state, env, net_adapter=None):

    backhaul_usage = 0
    n_subreqs = 1 + cfg.viewport
    transition = [None] * n_subreqs

    if req_state is None:
        return None, np.zeros(n_subreqs, dtype=np.int32), transition

    missing = env._missing_items(req_state)

    backhaul_usage = process_missing_layer(
        agent=agent,
        req_state=req_state,
        env=env,
        net_adapter=net_adapter,
        missing=missing,
        transition=transition,
        backhaul_usage=backhaul_usage,
        layer_idx=0,
        tile_idx=None,
        is_base=True,
        bytes_cost=12 * env.mec_cache.tile_size_bytes[0],
        log_key="base",
    )

    for idx in range(cfg.viewport):
        backhaul_usage = process_missing_layer(
            agent=agent,
            req_state=req_state,
            env=env,
            net_adapter=net_adapter,
            missing=missing,
            transition=transition,
            backhaul_usage=backhaul_usage,
            layer_idx=idx + 1,
            tile_idx=idx,
            is_base=False,
            bytes_cost=env.mec_cache.tile_size_bytes[1],
            log_key=f"enh_{idx}",
        )

    debugger.log("backhaul_usage", backhaul_usage)

    debugger.log("base_layer_miss", missing[0])
    for i, is_missing in enumerate(missing[1:], start=1):
        debugger.log(f"enh_layer_missing_{i}", is_missing)

    return None, missing, transition

def run_episode(episode, env, agent, net_adapter, cfg, metrics_dir, global_step_start):
    """Run one full training episode and persist step-level metrics."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0
    psnr_sum = 0.0

    global_step = global_step_start

    if cfg.has_warmup:
        env.warmup_phase(net_adapter, 1000)

    for step in range(cfg.max_steps):
        global_step += 1

        # --- Build State ---
        req_state = info.get("user_request", None)

        # --- Action Selection ---
        action, missing, transition = select_action(agent, req_state, env, net_adapter)

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        nxt_req = info["user_request"]

        reward_0 = info["reward_layer_0"]
        reward_1 = info["reward_layer_1"]

        reward = reward_0 + reward_1
        
        queued_update = False

        if missing[0] == 1 or transition[0] is not None:
            next_state_base = net_adapter.build_observation(nxt_req["video"])

            agent.remember(
                transition[0]['state'],
                transition[0]['action'],
                reward,
                next_state_base,
                done
            )
            queued_update = True

        for i in range(len(missing) - 1):
            if missing[i + 1] == 1 and transition[i + 1] is not None:
                next_state_enh = net_adapter.build_observation(nxt_req["video"], nxt_req["viewport"][i])
                agent.remember(
                    transition[i + 1]['state'],
                    transition[i + 1]['action'],
                    reward,
                    next_state_enh,
                    done
                )
                queued_update = True

        if queued_update:
            train_metrics = agent.train_step()

            if train_metrics is not None:
                save_step_metrics(
                    metrics_dir=metrics_dir,
                    episode=episode,
                    episode_step=step,
                    global_step=global_step,
                    reward=reward,
                    agent=agent,
                    train_metrics=train_metrics,
                )

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss
        psnr_sum += info.get("psnr", 0.0)

        if done:
            break

        debugger.log('cache_hits', bs_hits + e_hits)
        debugger.log('cache_misses', bs_miss + e_miss)

    return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses, global_step, psnr_sum / (step + 1)

def train(cfg):
    env = builders.build_environment(cfg)

    agent = A2CHeuWorker(cfg, debugger=debugger)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    metrics_dir = os.path.join(debug_path, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")
    print(f"Warmup Phase: {'Enabled' if cfg.has_warmup else 'Disabled'}")
    print(f"Users Session Length: {cfg.user_session_length}")
    print(agent)

    global_log_path = os.path.join(debug_path, "global.log")
    with open(global_log_path, "a", encoding="utf-8") as f:
        f.write(f"{pd.Timestamp.now().isoformat()} | {agent}\n")
        f.write("-" * 50 + "\n")

    global_step = 0

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss, global_step, psnr_rate = run_episode(
            episode, env, agent, net_adapter, cfg, metrics_dir, global_step
        )

        agent.update_epsilon() # Function used to clean buffer memory

        save_episode_metrics(
            metrics_dir=metrics_dir,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=agent,
        )

        network_params = {"episode": episode, "networks": {}}

        for net_name in ("q_network", "policy_net", "actor", "critic", "model", "network"):
            net = getattr(agent, net_name, None)
            if net is not None and hasattr(net, "state_dict"):
                network_params["networks"][net_name] = {
                    k: v.detach().cpu().tolist()
                    for k, v in net.state_dict().items()
                }

        if network_params["networks"]:
            json_path = os.path.join(metrics_dir, f"network_params_ep{episode}.json")
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(network_params, f, indent=2)

        debugger.log('lr', agent.scheduler.get_last_lr()[0])
        debugger.log('epsilon', agent.epsilon) 

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print(
            f"Episode {episode} | R: {int(total_reward)} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} | "
            f"EHR: {enh_hits / (enh_hits + enh_miss + 1e-9):.2f} | "
            f"PSNR: {psnr_rate:.2f} | "
            f"Time: {pd.Timestamp.now().strftime('%H:%M:%S')}"
        )
        print("-" * 50)

if __name__ == "__main__":
    train(cfg)

Starting training for 400 episodes... 2026-04-20_22-11
Warmup Phase: Enabled
Users Session Length: 60
A2CWorker(ActorLR=0.000100, CriticLR=0.002500)
BatchSize=128, Gamma=0.99
BufferSize=2000, GAE_lambda=0.95, Entropy_beta=0.01
Advantage_clip=5.0, Gradient_clip_norm=0.5
hiddens=(512, 1024, 512), Action Dim=251, State Dim=502
Episode 0 | R: 141325 | HR: 0.35 | BHR: 0.41 | EHR: 0.20 | PSNR: 32.38 | Time: 22:12:41
--------------------------------------------------
Episode 1 | R: 175665 | HR: 0.44 | BHR: 0.49 | EHR: 0.28 | PSNR: 33.14 | Time: 22:13:35
--------------------------------------------------
Episode 2 | R: 160980 | HR: 0.40 | BHR: 0.46 | EHR: 0.24 | PSNR: 32.80 | Time: 22:14:29
--------------------------------------------------


KeyboardInterrupt: 